In [25]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [26]:
load_dotenv()

MODEL = "openai/gpt-oss-20b"
openai = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")

In [27]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

### Tool

In [28]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [29]:
get_ticket_price("london")

Tool called for city london


'The price of a ticket to london is $799'

In [30]:
get_ticket_price("asdf")

Tool called for city asdf


'The price of a ticket to asdf is Unknown ticket price'

In [31]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [32]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

### OpenaAI to use our tool

In [33]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, tool_choice="auto")

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        tool_response = handle_tool_call(assistant_message)
        messages.append(tool_response)
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages
        )

    return response.choices[0].message.content

In [34]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]

    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    print("FUNCTION:", function_name)
    print("ARGUMENTS:", arguments)

    if function_name == "get_ticket_price":
        price_details = get_ticket_price(**arguments)

        return {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }

In [35]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


FUNCTION: get_ticket_price
ARGUMENTS: {'destination_city': 'London'}
Tool called for city London
FUNCTION: get_ticket_price
ARGUMENTS: {'destination_city': 'Somalia'}
Tool called for city Somalia
FUNCTION: get_ticket_price
ARGUMENTS: {'destination_city': 'Paris'}
Tool called for city Paris
